# Tutorial 02 — Voting Analysis: Party Discipline & Defection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kyusik-yang/assembly-explorer/blob/main/tutorials/02_voting_analysis.ipynb)

**Research question**: How cohesive are Korean legislative parties on floor votes, and when do members defect?

This notebook builds a voting analysis dataset from plenary roll-call records and computes standard political-science measures of party discipline.

**What you'll learn:**
- Collect per-member vote records for multiple bills in bulk
- Compute the **Rice index** of party cohesion per bill
- Identify individual **defectors** (members who cross the party line)
- Visualize party discipline over time and across issue areas
- Export a tidy panel dataset ready for regression analysis

**Prerequisite:** Complete Tutorial 01, or at minimum have an API key and the `AssemblyAPI` class available.

---

In [ ]:
# Install dependencies
!pip install httpx pandas plotly -q

In [ ]:
import os

try:
    from google.colab import userdata
    API_KEY = userdata.get('ASSEMBLY_API_KEY')
except Exception:
    API_KEY = os.getenv('ASSEMBLY_API_KEY', '')

if not API_KEY:
    API_KEY = input('Enter your ASSEMBLY_API_KEY: ').strip()

print('API key loaded:', bool(API_KEY))

In [ ]:
import httpx
import time

class AssemblyAPI:
    """Synchronous client for the Korean National Assembly Open API."""

    BASE = 'https://open.assembly.go.kr/portal/openapi'
    UNIT_CD = {'22':'100022','21':'100021','20':'100020',
               '19':'100019','18':'100018','17':'100017','16':'100016'}

    def __init__(self, api_key: str):
        self.key = api_key

    def _get(self, endpoint: str, **params) -> tuple[list[dict], int]:
        p = {'KEY': self.key, 'Type': 'json',
             **{k: v for k, v in params.items() if v is not None}}
        r = httpx.get(f'{self.BASE}/{endpoint}', params=p, timeout=30)
        r.raise_for_status()
        body = r.json().get(endpoint, [])
        if not body:
            return [], 0
        head  = body[0].get('head', [])
        total = int(head[0].get('list_total_count', 0)) if head else 0
        code  = (head[1].get('RESULT', {}) if len(head) > 1 else {}).get('CODE', '')
        if code == 'INFO-200':
            return [], 0
        rows = body[1].get('row', []) if len(body) > 1 else []
        return (rows if isinstance(rows, list) else [rows]), total

    def get_vote_results(self, age, bill_name=None, page=1, page_size=50):
        return self._get('ncocpgfiaoituanbr', AGE=age, BILL_NAME=bill_name,
                         pIndex=page, pSize=page_size)

    def get_member_votes(self, bill_id, age, page_size=300):
        return self._get('nojepdqqaweusdfbi', BILL_ID=bill_id, AGE=age,
                         pIndex=1, pSize=page_size)

    def search_bills(self, age, bill_name=None, proposer=None,
                     proc_result=None, committee=None,
                     dt_from=None, dt_to=None, page=1, page_size=100):
        return self._get('nzmimeepazxkubdpn', AGE=age, BILL_NAME=bill_name,
                         PROPOSER=proposer, PROC_RESULT=proc_result,
                         COMMITTEE=committee, STR_DT=dt_from, END_DT=dt_to,
                         pIndex=page, pSize=page_size)

api = AssemblyAPI(API_KEY)
print('Client ready.')

## 1. Collect voted bills

We start by retrieving a list of bills that received a plenary floor vote.
Each record includes bill-level totals (yes / no / abstain) and — crucially — a `BILL_ID`
that lets us fetch the individual member-level roll call.

In [ ]:
import pandas as pd

# Retrieve up to 50 recently voted bills from the 22nd Assembly
vote_rows, total_v = api.get_vote_results(age='22', page_size=50)
df_votes = pd.DataFrame(vote_rows)

for col in ['YES_TCNT', 'NO_TCNT', 'BLANK_TCNT', 'VOTE_TCNT']:
    if col in df_votes.columns:
        df_votes[col] = pd.to_numeric(df_votes[col], errors='coerce').fillna(0).astype(int)

print(f'Total voted bills: {total_v}  |  Retrieved: {len(df_votes)}')
display(df_votes[['BILL_NAME','PROC_RESULT_CD','YES_TCNT','NO_TCNT','BLANK_TCNT']].head(10))

## 2. Fetch per-member roll calls for multiple bills

We now fan out a `get_member_votes` call for each bill.
The result is a long-format panel: one row per (bill, member) pair.

> **Note on rate limits:** The API allows ~2 req/sec. We add a 0.5 s pause between calls
> to stay well within limits.

In [ ]:
# Collect member votes for the first N bills that have a BILL_ID
N_BILLS = 20   # Increase for a larger dataset (takes ~15 s for 20 bills)

bill_subset = df_votes[df_votes['BILL_ID'].notna()].head(N_BILLS)
all_mv = []

for i, (_, bill) in enumerate(bill_subset.iterrows()):
    bill_id   = bill['BILL_ID']
    bill_name = bill['BILL_NAME']
    rows, _   = api.get_member_votes(bill_id=bill_id, age='22')
    for r in rows:
        r['BILL_ID']   = bill_id
        r['BILL_NAME'] = bill_name
        r['PROC_DT']   = bill.get('PROC_DT', '')
    all_mv.extend(rows)
    print(f'  [{i+1}/{N_BILLS}] {bill_name[:35]}... ({len(rows)} members)', end='\r')
    time.sleep(0.5)

print(f'\nDone. Long-format rows: {len(all_mv)}')
df_mv = pd.DataFrame(all_mv)
display(df_mv[['BILL_NAME','HG_NM','POLY_NM','RESULT_VOTE_MOD']].head(8))

## 3. The Rice index of party cohesion

The **Rice index** is the standard measure of party cohesion for roll-call votes:

$$\text{Rice} = \frac{|\text{Yea} - \text{Nay}|}{\text{Yea} + \text{Nay}} \times 100$$

- **100** = all voting members agreed (perfectly unified)
- **0** = party split exactly 50–50
- Members who abstained or were absent are excluded from the denominator

In [ ]:
def compute_rice_index(vdf: pd.DataFrame, party_col='POLY_NM', vote_col='RESULT_VOTE_MOD') -> pd.DataFrame:
    """Compute Rice index per party from a member-vote DataFrame."""
    summary = []
    for party, grp in vdf.groupby(party_col):
        vc     = grp[vote_col].value_counts()
        yea    = int(vc.get('찬성', 0))
        nay    = int(vc.get('반대', 0))
        abst   = int(vc.get('기권', 0))
        absent = int(vc.get('불참', 0))
        denom  = yea + nay
        rice   = round(abs(yea - nay) / denom * 100, 1) if denom > 0 else None
        summary.append({
            'Party':      party,
            'Yea':        yea,
            'Nay':        nay,
            'Abstain':    abst,
            'Absent':     absent,
            'Rice Index': rice,
        })
    return (
        pd.DataFrame(summary)
        .sort_values('Rice Index', ascending=False, na_position='last')
        .fillna({'Rice Index': '—'})
    )


# Compute per-bill Rice index for each party
rice_records = []
for bill_id, grp in df_mv.groupby('BILL_ID'):
    bill_name = grp['BILL_NAME'].iloc[0]
    proc_dt   = grp['PROC_DT'].iloc[0]
    rice_df   = compute_rice_index(grp)
    for _, r in rice_df.iterrows():
        rice_records.append({
            'BILL_ID':    bill_id,
            'BILL_NAME':  bill_name,
            'PROC_DT':    proc_dt,
            'Party':      r['Party'],
            'Rice Index': r['Rice Index'],
            'Yea':        r['Yea'],
            'Nay':        r['Nay'],
            'Abstain':    r['Abstain'],
            'Absent':     r['Absent'],
        })

df_rice = pd.DataFrame(rice_records)
df_rice['Rice Index'] = pd.to_numeric(df_rice['Rice Index'], errors='coerce')

print('Rice index dataset shape:', df_rice.shape)
display(df_rice.head(12))

In [ ]:
import plotly.express as px

# Average Rice index by party across all sampled bills
avg_rice = (
    df_rice.dropna(subset=['Rice Index'])
    .groupby('Party', as_index=False)['Rice Index']
    .mean()
    .sort_values('Rice Index', ascending=False)
)

PARTY_COLORS = {
    '더불어민주당': '#004EA2',
    '국민의힘':     '#E61E2B',
    '개혁신당':     '#FF7210',
    '조국혁신당':   '#0095DA',
    '진보당':       '#D6001C',
    '무소속':       '#888888',
}

fig = px.bar(
    avg_rice, x='Party', y='Rice Index',
    title=f'Average Rice Index by party — {N_BILLS} bills, 22nd Assembly',
    color='Party',
    color_discrete_map=PARTY_COLORS,
    labels={'Rice Index': 'Avg. Rice Index (0–100)'},
    text='Rice Index',
)
fig.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig.update_layout(showlegend=False, yaxis_range=[0, 110], xaxis_tickangle=-30)
fig.show()

## 4. Identifying defectors

A **defector** is a member who votes against the plurality position of their own party.

For each (bill, party) pair we compute the **majority position** (the vote cast by the
most members), then flag anyone who voted differently.

In [ ]:
def label_defectors(vdf: pd.DataFrame) -> pd.DataFrame:
    """
    Add 'majority_vote' and 'defected' columns to a member-vote DataFrame.
    Only yea/nay votes are considered; abstain/absent are excluded from
    the majority calculation.
    """
    vdf = vdf.copy()
    active = vdf[vdf['RESULT_VOTE_MOD'].isin(['찬성', '반대'])]

    # Majority vote per (BILL_ID, POLY_NM)
    majority = (
        active.groupby(['BILL_ID', 'POLY_NM'])['RESULT_VOTE_MOD']
        .agg(lambda s: s.value_counts().index[0])
        .rename('majority_vote')
        .reset_index()
    )

    vdf = vdf.merge(majority, on=['BILL_ID', 'POLY_NM'], how='left')
    # Defected = voted in the minority on a yea/nay vote
    vdf['defected'] = (
        vdf['RESULT_VOTE_MOD'].isin(['찬성', '반대']) &
        (vdf['RESULT_VOTE_MOD'] != vdf['majority_vote'])
    )
    return vdf


df_mv = label_defectors(df_mv)

# Top defectors across all sampled bills
top_defectors = (
    df_mv[df_mv['defected']]
    .groupby(['HG_NM', 'POLY_NM'])
    .size()
    .reset_index(name='defection_count')
    .sort_values('defection_count', ascending=False)
    .head(20)
)

print('Top 20 defectors (out of sampled bills):')
display(top_defectors)

In [ ]:
# Defection rate by party
active_votes = df_mv[df_mv['RESULT_VOTE_MOD'].isin(['찬성', '반대'])]
defection_rate = (
    active_votes.groupby('POLY_NM')['defected']
    .mean()
    .mul(100).round(1)
    .reset_index(name='Defection Rate (%)')
    .sort_values('Defection Rate (%)', ascending=False)
)

fig = px.bar(
    defection_rate, x='POLY_NM', y='Defection Rate (%)',
    title=f'Defection rate by party — {N_BILLS} bills, 22nd Assembly',
    color='POLY_NM',
    color_discrete_map=PARTY_COLORS,
    labels={'POLY_NM': 'Party'},
    text='Defection Rate (%)',
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(showlegend=False, xaxis_tickangle=-30, yaxis_range=[0, 30])
fig.show()

## 5. Party discipline over time

Plot how the Rice index of each party evolves across the sample of bills,
ordered by vote date.

In [ ]:
import plotly.graph_objects as go

# Filter to the two largest parties for a cleaner plot
major_parties = df_rice.groupby('Party')['Rice Index'].count().nlargest(3).index.tolist()
df_time = (
    df_rice[df_rice['Party'].isin(major_parties)]
    .dropna(subset=['Rice Index', 'PROC_DT'])
    .sort_values('PROC_DT')
)

fig = go.Figure()
for party in major_parties:
    sub = df_time[df_time['Party'] == party]
    fig.add_scatter(
        x=sub['PROC_DT'], y=sub['Rice Index'],
        mode='lines+markers',
        name=party,
        line=dict(color=PARTY_COLORS.get(party, '#888888')),
        hovertemplate='%{x}<br>Rice Index: %{y}<extra>' + party + '</extra>',
    )

fig.update_layout(
    title='Rice Index over time (top 3 parties)',
    xaxis_title='Vote date', yaxis_title='Rice Index',
    yaxis_range=[0, 105], height=400,
)
fig.show()

## 6. Cross-party voting coalitions

Which bills attracted cross-party support (or opposition)?  
We look at bills where at least one party split from the others.

In [ ]:
# Dominant vote direction per party per bill
active2 = df_mv[df_mv['RESULT_VOTE_MOD'].isin(['찬성', '반대'])].copy()

party_positions = (
    active2.groupby(['BILL_ID', 'BILL_NAME', 'POLY_NM'])['RESULT_VOTE_MOD']
    .agg(lambda s: '찬성' if s.value_counts()['찬성'] >= s.value_counts().get('반대', 0) else '반대')
    .reset_index(name='position')
)

# Pivot: bills × parties
pivot = party_positions.pivot_table(
    index=['BILL_ID', 'BILL_NAME'],
    columns='POLY_NM',
    values='position',
    aggfunc='first',
).reset_index()

print('Party position matrix (찬성/반대 by bill):')
display(pivot.head(15))

In [ ]:
# Bills where major parties voted differently (contested bills)
major2 = ['더불어민주당', '국민의힘']
contested_cols = [c for c in major2 if c in pivot.columns]

if len(contested_cols) == 2:
    contested = pivot.dropna(subset=contested_cols)
    contested = contested[contested[contested_cols[0]] != contested[contested_cols[1]]]
    print(f'Contested bills (major parties split): {len(contested)}')
    display(contested[['BILL_NAME'] + contested_cols].head(10))
else:
    print('Need both 더불어민주당 and 국민의힘 in dataset.')

## 7. Export: tidy panel for regression

The panel below is analysis-ready:
- **Unit of observation**: (bill, member)
- **Outcome variables**: `RESULT_VOTE_MOD` (raw), `vote_binary` (1=Yea, 0=Nay), `defected`
- **Identifiers**: `BILL_ID`, `HG_NM` (member name), `POLY_NM` (party)

Export as UTF-8 with BOM for Stata/Excel compatibility with Korean text.

In [ ]:
# Add binary vote outcome
df_mv['vote_binary'] = df_mv['RESULT_VOTE_MOD'].map({'찬성': 1, '반대': 0})

# Select and rename columns for export
export_cols = [
    'BILL_ID', 'BILL_NAME', 'PROC_DT',
    'HG_NM', 'POLY_NM', 'ORIG_NM',
    'RESULT_VOTE_MOD', 'vote_binary', 'majority_vote', 'defected',
]
export_cols = [c for c in export_cols if c in df_mv.columns]
df_export = df_mv[export_cols].copy()

# Save
df_export.to_csv('vote_panel_22nd.csv', index=False, encoding='utf-8-sig')
df_rice.to_csv('rice_index_22nd.csv', index=False, encoding='utf-8-sig')

print(f'Saved vote_panel_22nd.csv ({df_export.shape})')
print(f'Saved rice_index_22nd.csv ({df_rice.shape})')

# In Colab: trigger download
try:
    from google.colab import files
    files.download('vote_panel_22nd.csv')
    files.download('rice_index_22nd.csv')
except Exception:
    pass

---

## Summary

| Output file | Contents |
|---|---|
| `vote_panel_22nd.csv` | Long-format (bill × member) roll-call panel with defection flag |
| `rice_index_22nd.csv` | Bill × party Rice index table |

**Key variables:**
- `RESULT_VOTE_MOD`: raw vote code — `찬성` (Yea), `반대` (Nay), `기권` (Abstain), `불참` (Absent)
- `vote_binary`: 1 = Yea, 0 = Nay (NaN for abstain/absent)
- `defected`: `True` if member voted against their party's majority position
- `Rice Index`: 0–100 party cohesion score per (bill, party)

**Next steps:**
- [Tutorial 03: Co-sponsorship Network Analysis](./03_network_analysis.ipynb)
- Load `vote_panel_22nd.csv` into Stata/R for logit/probit defection models
- Merge with member metadata (district, term, committee) for covariate analysis